# Patch precio v2 · Extractor robusto + trazabilidad

## Por qué este v2
El v1 (`patch_precio.ipynb`) solo cubría UN patrón de strike-through (`<s><span class="vc-number">`). Lentiamo usa varios markups distintos para el precio antiguo, así que el v1 dejó muchas filas sin corregir y, peor aún, las enmascaró rellenando con el precio anterior (que era el de oferta).

## Plan v2
1. Trabajamos desde `data/raw/lentiamo_graduadas_BACKUP_pre_patch.csv` (NO desde el CSV ya parcheado).
2. Re-fetcheamos cada URL.
3. Buscamos el **precio tachado** en MUCHOS sitios (tags, clases, data-attrs, JSON embebido).
4. Buscamos el **precio actual** en otros tantos.
5. Decidimos PVP con regla explícita y validación de coherencia.
6. Conservamos **AMBOS** precios + `en_oferta` + `patch_fallo` (trazabilidad).
7. Guardamos en archivo nuevo `lentiamo_graduadas_repatch.csv` — la canónica NO se toca.
8. Validación al final con tabla comparativa y muestreo aleatorio para revisar a mano.

## 1. Setup + carga del backup

In [1]:
import csv
import json
import logging
import random
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent
RAW = ROOT / 'data' / 'raw'
BACKUP    = RAW / 'lentiamo_graduadas_BACKUP_pre_patch.csv'
REPATCH   = RAW / 'lentiamo_graduadas_repatch.csv'
INTERMEDIO = RAW / 'lentiamo_graduadas_intermedio_repatch.csv'

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s',
                    datefmt='%H:%M:%S', force=True)
log = logging.getLogger('patch_v2')

assert BACKUP.exists(), f'Falta el backup: {BACKUP}'
df_backup = pd.read_csv(BACKUP)
print(f'Filas en backup: {len(df_backup)}')
df_backup.head(2)

Filas en backup: 2875


,url,marca,modelo,tipo,genero,material_montura,forma,tipo_montura,color,talla,ancho_lente,ancho_puente,largo_varilla,calibre_total,peso,polarizadas,precio
0,https://www.lentiamo.es/ray-ban-0rx5698-8109-5...,Ray-Ban,0RX5698 8109 56,graduadas,Unisex,Acetato,Aviador,Montura completa,Marrón,M,56.0,14.0,145.0,133.0,190.0,NaN,112.9
1,https://www.lentiamo.es/saint-laurent-sl-574-0...,Saint Laurent,SL 574 001 52,graduadas,Unisex,Pasta,Rectangulares,Montura completa,Negro,M,52.0,21.0,145.0,139.0,155.0,NaN,249.9


## 2. Funciones de fetch

In [2]:
HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/124.0.0.0 Safari/537.36'),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate',
}
DELAY_RANGE = (0.7, 1.8)
MAX_RETRIES = 3

PRECIO_MIN_VALIDO = 5.0     # menos de 5 € es ruido (IVA, envío)
PRECIO_MAX_VALIDO = 5000.0  # más de 5000 € imposible en gafas graduadas


def get_session():
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


def fetch(session, url):
    for i in range(MAX_RETRIES):
        try:
            r = session.get(url, timeout=20, allow_redirects=True)
            if r.status_code == 200:
                return r.text
            if r.status_code in (403, 429):
                time.sleep(5 * (i + 1))
                continue
            return None
        except requests.RequestException:
            time.sleep(2 * (i + 1))
    return None

## 3. Extractor de precios — múltiples estrategias

Diseñado para cubrir TODOS los patrones razonables de marcado:
- Tags: `<s>`, `<del>`
- Clases que sugieran precio antiguo (regex amplio)
- Atributos `data-*` típicos
- Claves en JSON embebido (scripts inline)

In [3]:
RE_PRECIO = re.compile(r'(\d{1,4}[.,]\d{2})')

RE_CLASE_TACHADO = re.compile(
    r'(strike|strikethrough|old|original|retail|regular|before|cross|compare|was|crossed)',
    re.IGNORECASE,
)

ATRIBS_TACHADO = [
    'data-original-price', 'data-old-price', 'data-list-price',
    'data-strike-price', 'data-was-price', 'data-regular-price',
    'data-retail-price', 'data-compare-price', 'data-msrp',
]

CLAVES_JSON_TACHADO = [
    'strikePrice', 'originalPrice', 'retailPrice', 'regularPrice',
    'listPrice', 'wasPrice', 'msrp', 'compareAtPrice',
]

CLAVES_JSON_ACTUAL = [
    'salePrice', 'currentPrice', 'price', 'finalPrice',
]


def parse_precio(texto):
    """Extrae el primer número con formato precio del texto. Devuelve float o None.
    Filtra precios absurdos (<5 € o >5000 €)."""
    if texto is None:
        return None
    s = str(texto).replace('\xa0', ' ').strip()
    m = RE_PRECIO.search(s)
    if not m:
        return None
    try:
        v = float(m.group(1).replace(',', '.'))
    except ValueError:
        return None
    if v < PRECIO_MIN_VALIDO or v > PRECIO_MAX_VALIDO:
        return None
    return v


def candidatos_tachado(soup, html):
    """Devuelve lista de precios candidatos a 'precio tachado'."""
    candidatos = []

    # 1. Tags <s> y <del> — cualquier texto dentro
    for tag in soup.find_all(['s', 'del']):
        # Primero busca un span numérico hijo (más específico)
        for span in tag.find_all(['span', 'div'], class_=re.compile(r'(vc-number|price|amount)', re.I)):
            v = parse_precio(span.get_text(' ', strip=True))
            if v: candidatos.append(v)
        # Si no hay span hijo, intenta con el propio texto del tag
        if not candidatos:
            v = parse_precio(tag.get_text(' ', strip=True))
            if v: candidatos.append(v)

    # 2. Cualquier elemento con clase que matchea precio-antiguo
    for el in soup.find_all(class_=RE_CLASE_TACHADO):
        v = parse_precio(el.get_text(' ', strip=True))
        if v: candidatos.append(v)

    # 3. Atributos data-* sospechosos
    for attr in ATRIBS_TACHADO:
        for el in soup.find_all(attrs={attr: True}):
            v = parse_precio(el[attr])
            if v: candidatos.append(v)

    # 4. JSON embebido en scripts (regex sobre HTML crudo)
    for clave in CLAVES_JSON_TACHADO:
        for m in re.finditer(rf'"{clave}"\s*:\s*"?(\d+[.,]\d+)"?', html):
            v = parse_precio(m.group(1))
            if v: candidatos.append(v)

    return candidatos


def candidatos_actual(soup, html):
    """Devuelve lista de precios candidatos a 'precio actual'."""
    candidatos = []

    # 1. itemprop="price" (microdata, muy fiable)
    for el in soup.find_all(attrs={'itemprop': 'price'}):
        v = parse_precio(el.get('content') or el.get_text(' ', strip=True))
        if v: candidatos.append(v)

    # 2. Meta tags Open Graph / product
    for prop in ('product:price:amount', 'og:price:amount'):
        m = soup.find('meta', attrs={'property': prop})
        if m and m.get('content'):
            v = parse_precio(m['content'])
            if v: candidatos.append(v)

    # 3. JSON embebido (claves típicas de precio actual)
    for clave in CLAVES_JSON_ACTUAL:
        for m in re.finditer(rf'"{clave}"\s*:\s*"?(\d+[.,]\d+)"?', html):
            v = parse_precio(m.group(1))
            if v: candidatos.append(v)

    # 4. Patrón "XX,XX €" en el HTML (último recurso)
    for m in re.finditer(r'(\d{1,4}[,\.]\d{2})\s*€', html):
        v = parse_precio(m.group(1))
        if v: candidatos.append(v)

    return candidatos


def precio_pvp_y_oferta(html):
    """Devuelve (precio_actual, precio_pvp, en_oferta, fallo).
    
    Reglas:
    - Si hay tachado y actual y tachado > actual → PVP=tachado, en_oferta=True.
    - Si solo hay actual → PVP=actual, en_oferta=False.
    - Si solo hay tachado (raro) → PVP=tachado, en_oferta=False.
    - Si no hay nada → fallo='parse'.
    
    Cuando hay varios candidatos del mismo tipo:
    - Para precio actual: el más bajo (más conservador, suele ser el real).
    - Para precio tachado: el más alto (el PVP suele ser mayor que cualquier ruido).
    """
    soup = BeautifulSoup(html, 'html.parser')
    cand_t = candidatos_tachado(soup, html)
    cand_a = candidatos_actual(soup, html)

    tachado = max(cand_t) if cand_t else None
    actual  = min(cand_a) if cand_a else None

    # Validación de coherencia: tachado solo cuenta como PVP si > actual
    if tachado is not None and actual is not None:
        if tachado > actual:
            return actual, tachado, True, ''
        else:
            # tachado <= actual → ruido (cross-out de envíos, etc.)
            return actual, actual, False, ''
    if actual is not None:
        return actual, actual, False, ''
    if tachado is not None:
        return tachado, tachado, False, ''
    return None, None, False, 'parse'

### Test rápido del extractor

In [4]:
# Test 1: HTML con <s> y itemprop (caso típico Lentiamo, en oferta)
html_oferta = '<s><span class="vc-number">45,00&nbsp;€</span></s><span itemprop="price" content="38.25"></span>'
print('Oferta clásica:', precio_pvp_y_oferta(html_oferta))
# Esperado: (38.25, 45.0, True, '')

# Test 2: HTML sin oferta (solo precio normal)
html_normal = '<span itemprop="price" content="120.00"></span>'
print('Sin oferta:    ', precio_pvp_y_oferta(html_normal))
# Esperado: (120.0, 120.0, False, '')

# Test 3: HTML con <del> (variante)
html_del = '<del>89,90 €</del><span itemprop="price" content="75.00"></span>'
print('Con <del>:     ', precio_pvp_y_oferta(html_del))
# Esperado: (75.0, 89.9, True, '')

# Test 4: HTML con clase 'old-price'
html_clase = '<span class="old-price">200,00 €</span><span itemprop="price" content="150.00"></span>'
print('Clase old:     ', precio_pvp_y_oferta(html_clase))
# Esperado: (150.0, 200.0, True, '')

# Test 5: HTML con data-original-price
html_data = '<div data-original-price="99.90"></div><span itemprop="price" content="79.90"></span>'
print('data-attr:     ', precio_pvp_y_oferta(html_data))
# Esperado: (79.9, 99.9, True, '')

# Test 6: HTML con JSON strikePrice
html_json = '<script>{"strikePrice":150.00,"price":99.90}</script>'
print('JSON keys:     ', precio_pvp_y_oferta(html_json))
# Esperado: (99.9, 150.0, True, '')

# Test 7: HTML vacío
print('Vacío:         ', precio_pvp_y_oferta('<html><body></body></html>'))
# Esperado: (None, None, False, 'parse')

Oferta clásica: (38.25, 45.0, True, '')
Sin oferta:     (120.0, 120.0, False, '')
Con <del>:      (75.0, 89.9, True, '')
Clase old:      (150.0, 200.0, True, '')
data-attr:      (79.9, 99.9, True, '')
JSON keys:      (99.9, 150.0, True, '')
Vacío:          (None, None, False, 'parse')


## 4. Re-fetch incremental con resume

In [5]:
# Si hay intermedio, retomamos
if INTERMEDIO.exists():
    df = pd.read_csv(INTERMEDIO)
    if {'precio_actual_v2', 'precio_pvp_v2', 'patch_fallo'}.issubset(df.columns) and len(df) == len(df_backup):
        log.info('Retomando desde intermedio: %d / %d procesadas',
                 df['patch_fallo'].notna().sum(), len(df))
    else:
        log.info('Intermedio incompatible, empiezo de cero')
        df = df_backup.copy()
        df['precio_actual_v2'] = pd.NA
        df['precio_pvp_v2'] = pd.NA
        df['en_oferta_v2'] = pd.NA
        df['patch_fallo'] = pd.NA
else:
    df = df_backup.copy()
    df['precio_actual_v2'] = pd.NA
    df['precio_pvp_v2'] = pd.NA
    df['en_oferta_v2'] = pd.NA
    df['patch_fallo'] = pd.NA

session = get_session()
n_procesadas = 0
n_ofertas = 0
n_fallos_fetch = 0
n_fallos_parse = 0

for i, row in df.iterrows():
    if pd.notna(row['patch_fallo']):
        continue  # ya procesada (incluido el caso de fallo previo, no reintentamos)
    url = row['url']
    html = fetch(session, url)
    time.sleep(random.uniform(*DELAY_RANGE))
    if not html:
        df.at[i, 'patch_fallo'] = 'fetch'
        n_fallos_fetch += 1
        continue
    actual, pvp, oferta, fallo = precio_pvp_y_oferta(html)
    if fallo:
        df.at[i, 'patch_fallo'] = fallo
        n_fallos_parse += 1
        continue
    df.at[i, 'precio_actual_v2'] = actual
    df.at[i, 'precio_pvp_v2']    = pvp
    df.at[i, 'en_oferta_v2']     = bool(oferta)
    df.at[i, 'patch_fallo']      = ''   # OK
    n_procesadas += 1
    if oferta:
        n_ofertas += 1

    if (n_procesadas + n_fallos_fetch + n_fallos_parse) % 50 == 0:
        df.to_csv(INTERMEDIO, index=False)
        log.info('Progreso: %d OK | %d ofertas | %d fetch_fail | %d parse_fail',
                 n_procesadas, n_ofertas, n_fallos_fetch, n_fallos_parse)

df.to_csv(INTERMEDIO, index=False)
log.info('FIN. OK=%d ofertas=%d fetch_fail=%d parse_fail=%d',
         n_procesadas, n_ofertas, n_fallos_fetch, n_fallos_parse)

10:49:48 [INFO] Progreso: 48 OK | 0 ofertas | 0 fetch_fail | 2 parse_fail
10:51:38 [INFO] Progreso: 98 OK | 5 ofertas | 0 fetch_fail | 2 parse_fail
10:53:28 [INFO] Progreso: 148 OK | 10 ofertas | 0 fetch_fail | 2 parse_fail
10:55:15 [INFO] Progreso: 198 OK | 16 ofertas | 0 fetch_fail | 2 parse_fail
10:57:07 [INFO] Progreso: 248 OK | 22 ofertas | 0 fetch_fail | 2 parse_fail
10:59:06 [INFO] Progreso: 297 OK | 31 ofertas | 0 fetch_fail | 3 parse_fail
11:01:12 [INFO] Progreso: 347 OK | 38 ofertas | 0 fetch_fail | 3 parse_fail
11:03:09 [INFO] Progreso: 397 OK | 45 ofertas | 0 fetch_fail | 3 parse_fail
11:05:06 [INFO] Progreso: 447 OK | 60 ofertas | 0 fetch_fail | 3 parse_fail
11:06:50 [INFO] Progreso: 497 OK | 71 ofertas | 0 fetch_fail | 3 parse_fail
11:08:40 [INFO] Progreso: 547 OK | 83 ofertas | 0 fetch_fail | 3 parse_fail
11:10:29 [INFO] Progreso: 597 OK | 96 ofertas | 0 fetch_fail | 3 parse_fail
11:12:23 [INFO] Progreso: 647 OK | 102 ofertas | 0 fetch_fail | 3 parse_fail
11:14:12 [INFO]

## 5. Validación previa antes de generar el CSV final

In [6]:
# Distribución de patch_fallo
print('Distribución de patch_fallo:')
print(df['patch_fallo'].value_counts(dropna=False))
print()
print(f'Filas con precio_pvp_v2 nulo: {df["precio_pvp_v2"].isna().sum()}')
print(f'Filas con en_oferta_v2=True:  {(df["en_oferta_v2"] == True).sum()}')
print(f'Filas con en_oferta_v2=False: {(df["en_oferta_v2"] == False).sum()}')

Distribución de patch_fallo:
patch_fallo
         2872
parse       3
Name: count, dtype: int64

Filas con precio_pvp_v2 nulo: 3
Filas con en_oferta_v2=True:  928
Filas con en_oferta_v2=False: 1944


In [8]:
# Tabla comparativa solicitada: backup vs v2
comp = df[['url', 'marca', 'modelo', 'precio',
           'precio_actual_v2', 'precio_pvp_v2', 'en_oferta_v2', 'patch_fallo']].copy()

comp.rename(columns={'precio': 'precio_backup'}, inplace=True)

comp['precio_backup'] = pd.to_numeric(comp['precio_backup'], errors='coerce')
comp['precio_actual_v2'] = pd.to_numeric(comp['precio_actual_v2'], errors='coerce')
comp['precio_pvp_v2'] = pd.to_numeric(comp['precio_pvp_v2'], errors='coerce')

comp['diff'] = comp['precio_pvp_v2'] - comp['precio_backup']
comp['diff_%'] = ((comp['diff'] / comp['precio_backup']) * 100).round(1)

# Ordenar para ver primero los cambios más grandes
comp_ordenada = (
    comp.assign(abs_diff=comp['diff'].abs())
        .sort_values('abs_diff', ascending=False)
        .drop(columns='abs_diff')
)

comp_ordenada.head(20)

,url,marca,modelo,precio_backup,precio_actual_v2,precio_pvp_v2,en_oferta_v2,patch_fallo,diff,diff_%
37,https://www.lentiamo.es/oliver-peoples-kinston...,Oliver Peoples,0OV1368T 5254 44,479.9,52.00,52.00,False,,-427.90,-89.2
1022,https://www.lentiamo.es/blackfin-mayfair-bf108...,Blackfin,BF1088 1622 20 52,459.9,45.00,45.00,False,,-414.90,-90.2
292,https://www.lentiamo.es/tom-ford-ft5994-b-001-...,Tom Ford,FT5994-B 001 56,419.9,45.00,45.00,False,,-374.90,-89.3
2858,https://www.lentiamo.es/tom-ford-ft6040-b-001-...,Tom Ford,FT6040-B 001 53,399.9,45.00,45.00,False,,-354.90,-88.7
996,https://www.lentiamo.es/blackfin-knightsbridge...,Blackfin,BF1089 1844 19 54,399.9,52.00,52.00,False,,-347.90,-87.0
1033,https://www.lentiamo.es/blackfin-mendocino-bf1...,Blackfin,BF1092 1707 15 54,379.9,45.00,45.00,False,,-334.90,-88.2
1008,https://www.lentiamo.es/blackfin-hayle-bf948-1...,Blackfin,BF948 1407 20 55,379.9,45.00,45.00,False,,-334.90,-88.2
1013,https://www.lentiamo.es/blackfin-aero-bf1076-1...,Blackfin,BF1076 1809 19GM 55,379.9,45.00,45.00,False,,-334.90,-88.2
1002,https://www.lentiamo.es/blackfin-sandpoint-bf1...,Blackfin,BF1117 1877 15 58,409.9,95.99,95.99,False,,-313.91,-76.6
414,https://www.lentiamo.es/gucci-gg0392o-002-51.html,Gucci,GG0392O 002 51,357.9,45.00,45.00,False,,-312.90,-87.4


In [9]:
comp_ordenada

,url,marca,modelo,precio_backup,precio_actual_v2,precio_pvp_v2,en_oferta_v2,patch_fallo,diff,diff_%
37,https://www.lentiamo.es/oliver-peoples-kinston...,Oliver Peoples,0OV1368T 5254 44,479.90,52.00,52.00,False,,-427.9,-89.2
1022,https://www.lentiamo.es/blackfin-mayfair-bf108...,Blackfin,BF1088 1622 20 52,459.90,45.00,45.00,False,,-414.9,-90.2
292,https://www.lentiamo.es/tom-ford-ft5994-b-001-...,Tom Ford,FT5994-B 001 56,419.90,45.00,45.00,False,,-374.9,-89.3
2858,https://www.lentiamo.es/tom-ford-ft6040-b-001-...,Tom Ford,FT6040-B 001 53,399.90,45.00,45.00,False,,-354.9,-88.7
996,https://www.lentiamo.es/blackfin-knightsbridge...,Blackfin,BF1089 1844 19 54,399.90,52.00,52.00,False,,-347.9,-87.0
...,...,...,...,...,...,...,...,...,...,...
2789,https://www.lentiamo.es/vogue-0vo4231-848-53.html,Vogue,0VO4231 848 53,69.99,69.99,69.99,False,,0.0,0.0
16,https://www.lentiamo.es/leader-spray-limpiador...,NaN,"Spray limpiador de gafas Lentiamo 29,5 ml",3.99,NaN,NaN,<NA>,parse,NaN,NaN
33,https://www.lentiamo.es/solunate-multi-purpose...,NaN,Solunate Multi-Purpose 50 ml con estuche,3.89,NaN,NaN,<NA>,parse,NaN,NaN
47,https://www.lentiamo.es/ray-ban-0rx8908-5196.html,Ray-Ban,Ray-Ban 0RX8908 5196,NaN,45.00,45.00,False,,NaN,NaN


In [10]:
# Resumen de cambios
print('Filas cuyo PVP cambió respecto al backup:')
n_cambios = (comp['diff'].abs() > 0.01).sum()
print(f'  · {n_cambios} de {len(comp)} ({n_cambios/len(comp)*100:.1f} %)')
print()
print('Cruce en_oferta_v2 × cambió_precio:')
tabla = pd.crosstab(
    comp['en_oferta_v2'].fillna('NaN'),
    (comp['diff'].abs() > 0.01).map({True: 'cambió', False: 'igual'}),
)
print(tabla)
print()
print('Estadísticas de los descuentos detectados (solo en_oferta=True):')
print(comp[comp['en_oferta_v2'] == True]['diff_%'].describe().round(1))

Filas cuyo PVP cambió respecto al backup:
  · 2643 de 2875 (91.9 %)

Cruce en_oferta_v2 × cambió_precio:
diff          cambió  igual
en_oferta_v2               
False           1716    228
True             927      1
NaN                0      3

Estadísticas de los descuentos detectados (solo en_oferta=True):
count    928.0
mean      87.3
std       69.9
min        0.0
25%       34.3
50%       71.4
75%      121.3
max      397.6
Name: diff_%, dtype: float64


In [11]:
# Muestreo aleatorio para revisar a mano contra la web
# 15 productos en oferta + 15 sin oferta
muestras = pd.concat([
    comp[comp['en_oferta_v2'] == True].sample(min(15, (comp['en_oferta_v2'] == True).sum()), random_state=42),
    comp[comp['en_oferta_v2'] == False].sample(min(15, (comp['en_oferta_v2'] == False).sum()), random_state=42),
])
print('Muestreo para revisión manual (compara con la web):')
print('Para cada URL, abre la página y verifica:')
print('  · Si en_oferta_v2=True, debería verse precio_pvp_v2 tachado y precio_actual_v2 como precio actual.')
print('  · Si en_oferta_v2=False, debería verse solo un precio = precio_pvp_v2 = precio_actual_v2.')
print()
muestras[['url', 'marca', 'precio_backup', 'precio_actual_v2', 'precio_pvp_v2', 'en_oferta_v2']]

Muestreo para revisión manual (compara con la web):
Para cada URL, abre la página y verifica:
  · Si en_oferta_v2=True, debería verse precio_pvp_v2 tachado y precio_actual_v2 como precio actual.
  · Si en_oferta_v2=False, debería verse solo un precio = precio_pvp_v2 = precio_actual_v2.



,url,marca,precio_backup,precio_actual_v2,precio_pvp_v2,en_oferta_v2
2507,https://www.lentiamo.es/sea2see-sirocco-c00-52...,Sea2See,94.95,94.95,189.90,True
2145,https://www.lentiamo.es/prada-linea-rossa-0ps-...,Prada Linea Rossa,102.12,102.12,256.90,True
297,https://www.lentiamo.es/nano-vista-loading-3-0...,Nano Vista,99.99,99.99,119.90,True
1364,https://www.lentiamo.es/esprit-et33505-543-53....,Esprit,66.88,66.88,102.90,True
915,https://www.lentiamo.es/active-memory-f0172450...,Centrostyle S.p.A,89.99,89.99,93.99,True
1265,https://www.lentiamo.es/dsquared2-d2-0069-pjp-...,Dsquared2,80.56,80.56,138.90,True
665,https://www.lentiamo.es/michael-kors-encino-0m...,Michael Kors,99.99,99.99,135.90,True
1724,https://www.lentiamo.es/marc-jacobs-marc-704-n...,Marc Jacobs,84.74,84.74,203.90,True
1683,https://www.lentiamo.es/jimmy-choo-0jc3026-500...,Jimmy Choo,127.72,127.72,179.90,True
1630,https://www.lentiamo.es/jaguar-32708-5282-21-4...,Jaguar,79.07,79.07,259.90,True


In [12]:
# Filas con fallo de parse — son las que necesitan investigación manual
fallos_parse = df[df['patch_fallo'] == 'parse']
if len(fallos_parse) > 0:
    print(f'⚠ {len(fallos_parse)} filas con fallo de parse — abre estas URLs y dime qué ves:')
    print(fallos_parse[['url', 'marca', 'modelo']].head(20))
else:
    print('✅ Ningún fallo de parse — el extractor cubrió todos los HTML descargados.')

⚠ 3 filas con fallo de parse — abre estas URLs y dime qué ves:
                                                   url       marca  \
16   https://www.lentiamo.es/leader-spray-limpiador...         NaN   
33   https://www.lentiamo.es/solunate-multi-purpose...         NaN   
255  https://www.lentiamo.es/nano-vista-gran-turism...  Nano Vista   

                                        modelo  
16   Spray limpiador de gafas Lentiamo 29,5 ml  
33    Solunate Multi-Purpose 50 ml con estuche  
255                 Gran Turismo SC NAO7809 46  


In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
RAW = ROOT / 'data' / 'raw'

backup = pd.read_csv(RAW / 'lentiamo_graduadas_BACKUP_pre_patch.csv')
v2 = pd.read_csv(RAW / 'lentiamo_graduadas_intermedio_repatch.csv')

df = backup.merge(
    v2[['url', 'precio_actual_v2', 'precio_pvp_v2', 'en_oferta_v2', 'patch_fallo']],
    on='url',
    how='left'
)

for c in ['precio', 'precio_actual_v2', 'precio_pvp_v2']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

mask_oferta_valida = (
    df['precio'].notna() &
    df['precio_actual_v2'].notna() &
    df['precio_pvp_v2'].notna() &
    np.isclose(df['precio_actual_v2'], df['precio'], atol=0.01) &
    (df['precio_pvp_v2'] > df['precio'] + 0.01)
)

mask_v2_toco = (
    df['precio_actual_v2'].notna() &
    df['precio_pvp_v2'].notna()
)

df['precio_actual'] = df['precio']
df['precio_pvp'] = df['precio']
df.loc[mask_oferta_valida, 'precio_pvp'] = df.loc[mask_oferta_valida, 'precio_pvp_v2']

df['en_oferta'] = mask_oferta_valida
df['patch_v2_descartado'] = mask_v2_toco & ~mask_oferta_valida

In [14]:
print('Filas totales:', len(df))
print('Ofertas válidas recuperadas:', df['en_oferta'].sum())
print('Filas donde v2 se descarta:', df['patch_v2_descartado'].sum())
print('Filas sin tocar:', (~df['en_oferta'] & ~df['patch_v2_descartado']).sum())

Filas totales: 2875
Ofertas válidas recuperadas: 919
Filas donde v2 se descarta: 1953
Filas sin tocar: 3


In [15]:
df.loc[df['en_oferta'], [
    'marca', 'modelo', 'precio', 'precio_actual_v2', 'precio_pvp_v2',
    'precio_actual', 'precio_pvp', 'url'
]].sample(20, random_state=42)

,marca,modelo,precio,precio_actual_v2,precio_pvp_v2,precio_actual,precio_pvp,url
519,Armani Exchange,0AX3104 8160 53,66.99,66.99,204.90,66.99,204.90,https://www.lentiamo.es/armani-exchange-0ax310...
769,Arnette,0AN7179 01 54,64.99,64.99,209.90,64.99,209.90,https://www.lentiamo.es/arnette-leonardo-0an71...
1160,David Beckham,DB 7020 2W8 20 51,89.45,89.45,256.90,89.45,256.90,https://www.lentiamo.es/david-beckham-db-7020-...
1194,Dolce & Gabbana,0DG1358 01 54,134.13,134.13,219.90,134.13,219.90,https://www.lentiamo.es/dolce-gabbana-0dg1358-...
2386,Ray-Ban,0RX2242V 8292 50,80.94,80.94,249.90,80.94,249.90,https://www.lentiamo.es/ray-ban-wayfarer-oval-...
1791,Max Mara,MM 5115 001 16 52,85.33,85.33,116.90,85.33,116.90,https://www.lentiamo.es/maxmara-mm-5115-001-16...
2744,Tommy Hilfiger,TH 2165/F 807 21 52,67.49,67.49,134.90,67.49,134.90,https://www.lentiamo.es/tommy-hilfiger-th-2165...
1797,Max Mara,MM 5114 001 16 54,76.26,76.26,143.90,76.26,143.90,https://www.lentiamo.es/maxmara-mm-5114-001-16...
2227,Puma,PU0096O 006 56,81.99,81.99,209.90,81.99,209.90,https://www.lentiamo.es/puma-pu0096o-006-56.html
1669,Jaguar,33827 1182 18 53,84.95,84.95,209.90,84.95,209.90,https://www.lentiamo.es/jaguar-33827-1182-18-5...


## 6. Generar el CSV repatch (sin sobrescribir la canónica)

In [ ]:
# Construir el DataFrame final con AMBOS precios + en_oferta + patch_fallo
df_final = df.copy()

# precio (original) → precio_actual (lo que paga el cliente hoy)
# Para filas con patch_fallo, mantenemos el del backup como precio_actual
df_final['precio_actual'] = df_final['precio_actual_v2'].fillna(df_final['precio'])
df_final['precio_pvp']    = df_final['precio_pvp_v2'].fillna(df_final['precio'])
df_final['en_oferta']     = df_final['en_oferta_v2'].fillna(False).astype(bool)

# Para usar como TARGET en el modelo, sobreescribimos `precio` con `precio_pvp`
# (más robusto que el precio actual, no depende de promociones temporales)
df_final['precio'] = df_final['precio_pvp']

# Quitar columnas auxiliares
df_final = df_final.drop(columns=['precio_actual_v2', 'precio_pvp_v2', 'en_oferta_v2'])

# Reordenar (mover los nuevos cerca de precio para legibilidad)
cols = [c for c in df_final.columns if c not in ['precio', 'precio_actual', 'precio_pvp', 'en_oferta', 'patch_fallo']]
cols += ['precio_actual', 'precio_pvp', 'en_oferta', 'patch_fallo', 'precio']
df_final = df_final[cols]

df_final.head(3)

In [ ]:
df_final.to_csv(REPATCH, index=False)
print(f'✅ Guardado: {REPATCH}')
print(f'   {len(df_final)} filas, {len(df_final.columns)} columnas')
print()
print('Columnas finales:')
for c in df_final.columns:
    print(f'  · {c}')

## 7. Plan de validación (acciones para Juan Antonio)

**ANTES de promover este CSV a canónico, hacer:**

1. **Mira la celda del muestreo aleatorio (sec. 5).** Abre 5-10 de esas URLs en el navegador y verifica que:
   - Las marcadas con `en_oferta_v2=True` muestran realmente PVP tachado + precio actual.
   - Las marcadas con `en_oferta_v2=False` muestran un único precio = al `precio_pvp_v2`.
2. **Mira las filas con `patch_fallo='parse'`.** Si son pocas (<10), abre y diagnóstica a mano qué markup raro tienen.
3. **Compara el % de productos en oferta** (debería ser razonable, 20-40 % en una óptica con políticas habituales).

**SI todo cuadra:**
1. Reemplaza la canónica con el repatch:
   ```python
   import shutil
   shutil.copy(REPATCH, RAW / 'lentiamo_graduadas.csv')
   ```
2. Vuelve a ejecutar `02_LimpiezaEDA.ipynb` para regenerar `data/processed/...` y los splits.
3. En el `02`, considera añadir `en_oferta` y opcionalmente `precio_actual` como features (este último te dice cuánto descuenta cada producto, info de negocio útil).

**SI no cuadra (muestreo manual revela errores):**
- Pega aquí la URL conflictiva + lo que ves en la web vs lo que dice el CSV.
- Investigamos el HTML de esa URL específica y añadimos un patrón nuevo al extractor.
- Re-ejecutas este notebook (gracias al resume incremental, solo procesa lo que falte).